## 10.Vector Stores

<pre>
A vector database is a database designed to store and search vectors/embeddings efficiently.
</pre>

Note: pip install langchain-chroma chromadb

In [2]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 649.05it/s]


In [3]:
documents = [
    "SQL indexing can make database queries faster.",
    "Query optimization reduces unnecessary database operations.",
    "Database tuning improves the overall performance of a database.",
    "Python is useful for data analysis.",
    "Machine learning models can be trained using Python."
]

<pre>
Chroma works nicely with LangChain Document objects, so let's convert them.
</pre>

In [4]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="SQL indexing can make database queries faster.",
        metadata={"topic": "database", "source": "database_notes"}
    ),
    
    Document(
        page_content="Query optimization reduces unnecessary database operations.",
        metadata={"topic": "database", "source": "database_notes"}
    ),
    
    Document(
        page_content="Database tuning improves the overall performance of a database.",
        metadata={"topic": "database", "source": "database_notes"}
    ),
    
    Document(
        page_content="Python is useful for data analysis.",
        metadata={"topic": "python", "source": "python_notes"}
    ),
    
    Document(
        page_content="Machine learning models can be trained using Python.",
        metadata={"topic": "machine_learning", "source": "ml_notes"}
    )
]

In [5]:
# create chroma vector store

vectorstore=Chroma.from_documents(documents=docs,embedding=embeddings,collection_name="semantic_search_demo")

In [6]:
query="How can i improve database performance?"
results=vectorstore.similarity_search(query)

In [7]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
Database tuning improves the overall performance of a database.
Metadata: {'source': 'database_notes', 'topic': 'database'}

--- Result 2 ---
Query optimization reduces unnecessary database operations.
Metadata: {'source': 'database_notes', 'topic': 'database'}

--- Result 3 ---
SQL indexing can make database queries faster.
Metadata: {'source': 'database_notes', 'topic': 'database'}

--- Result 4 ---
Python is useful for data analysis.
Metadata: {'topic': 'python', 'source': 'python_notes'}


In [8]:
# Get similarity scores

results = vectorstore.similarity_search_with_score(query)

for doc, score in results:
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print()

Score: 0.6209
Database tuning improves the overall performance of a database.

Score: 0.8190
Query optimization reduces unnecessary database operations.

Score: 0.9365
SQL indexing can make database queries faster.

Score: 1.6694
Python is useful for data analysis.



<pre>
With Chroma, the returned value is typically a distance, not necessarily the same cosine-similarity score.
For Chroma's default distance metric, lower distance generally means more similar.
</pre>

In [9]:
# Let's use top k results
results=vectorstore.similarity_search(query,k=2)
for doc in results:
    print(doc.page_content)

Database tuning improves the overall performance of a database.
Query optimization reduces unnecessary database operations.


In [11]:
# Metadata filtering

results=vectorstore.similarity_search(query,k=3,filter={"topic":"database"})

In [12]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print()

Database tuning improves the overall performance of a database.
{'topic': 'database', 'source': 'database_notes'}

Query optimization reduces unnecessary database operations.
{'topic': 'database', 'source': 'database_notes'}

SQL indexing can make database queries faster.
{'source': 'database_notes', 'topic': 'database'}



### ChromaDB + PDF — Full Practical Implementation

In [14]:
#load pdf
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader('text.pdf')
documents=loader.load()
len(documents)

Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)


15

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=splitter.split_documents(documents)
print("Number of chunks:",len(chunks))

Number of chunks: 37


In [18]:
for chunk in chunks:
    print("Chunk:",chunk.page_content)
    print("MetaData:",chunk.metadata)
    print("=============================")

Chunk: The Making of Indian Constitution (I)
MetaData: {'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20260106092821Z00'00'", 'title': 'THe Making of Indian Constitution I', 'moddate': "D:20260106092821Z00'00'", 'source': 'text.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}
Chunk: Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees
MetaData: {'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20260106092821Z00'00'", 'title': 'THe Making of Indian Constitution I', 'moddate': "D:20260106092821Z00'00'", 'source': 'text.pdf', 'total_pages': 15, 'page': 1, 'page_label': '2'}
Chunk: Introduction
•The power was centralised under the Government of India Act 1858 - 
gradually “shared” through limited reforms - Minto-Morley Reforms (1909); 
Montague-Chelmsford Reforms (1919); Government of India Act (1935) - 
but the power wa

In [20]:
# create our embedding model
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 917.53it/s]


In [ ]:
 # Create chromaDB
from langchain_chroma import Chroma

# insert documents
chromadb=Chroma.from_documents(documents=chunks,embedding=embeddings,collection_name="pdf_semantic_search")

In [22]:
# SIMILARITY SEARCH
query="Analyze the tension between the 'electoral under-representation' and the 'sociological inclusiveness' of the Constituent Assembly. Which groups were included to ensure broad legitimacy?"

results=chromadb.similarity_search(query)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
legislatures, elected under the 1935 Act 
•Tension between electoral under-representation (limited franchise, princely nomination) and sociological 
inclusiveness.  
•The election of Constituent Assembly members was by the provincial legislatures, the Congress high 
command adopted much the same policy; i.e., let the provincial machinery select its own members; this 
preserved the diversity present in the legislatures and added to it in an important way.
Metadata: {'creator': 'Keynote', 'page': 7, 'source': 'text.pdf', 'total_pages': 15, 'creationdate': "D:20260106092821Z00'00'", 'title': 'THe Making of Indian Constitution I', 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'page_label': '8', 'moddate': "D:20260106092821Z00'00'"}

--- Result 2 ---
Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees
Metadata: {'creator': 'Keynote', 'source': 'text.pdf', 'total_pages': 15, 'moddate': "D:20260106092821Z00'

In [25]:
# top-k search

results=chromadb.similarity_search(query,k=3)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
legislatures, elected under the 1935 Act 
•Tension between electoral under-representation (limited franchise, princely nomination) and sociological 
inclusiveness.  
•The election of Constituent Assembly members was by the provincial legislatures, the Congress high 
command adopted much the same policy; i.e., let the provincial machinery select its own members; this 
preserved the diversity present in the legislatures and added to it in an important way.

--- Result 2 ---
Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees

--- Result 3 ---
• There was no universal adult franchise and the electorate for the provincial legislatures was restricted by 
property, education and tax qualifications.As there was communal allocation of seats and princely nominations, 
some communities were structurally over represented or under represented. 
•The legitimacy for the members came from their deeply imbedded work in the national move

In [27]:
# similarity scores

results = chromadb.similarity_search_with_score(
    query,
    k=3
)

for doc, score in results:
    print("Score:", score)
    print("Text:", doc.page_content[:300])
    print("Metadata:", doc.metadata)
    print()

Score: 0.6959396600723267
Text: legislatures, elected under the 1935 Act 
•Tension between electoral under-representation (limited franchise, princely nomination) and sociological 
inclusiveness.  
•The election of Constituent Assembly members was by the provincial legislatures, the Congress high 
command adopted much the same pol
Metadata: {'page': 7, 'page_label': '8', 'title': 'THe Making of Indian Constitution I', 'creator': 'Keynote', 'creationdate': "D:20260106092821Z00'00'", 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'total_pages': 15, 'moddate': "D:20260106092821Z00'00'", 'source': 'text.pdf'}

Score: 0.730057954788208
Text: Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees
Metadata: {'title': 'THe Making of Indian Constitution I', 'total_pages': 15, 'moddate': "D:20260106092821Z00'00'", 'source': 'text.pdf', 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creationdate': "D:2026010609282

<pre>
⚠️ Important:

With Chroma's default setup, this value is generally a distance, not a cosine-similarity score.

So:

smaller distance
      ↓
more similar
</pre>

<pre>
Don't confuse this with our earlier manual cosine similarity where:

higher similarity
      ↓
more similar
</pre>

In [30]:
# metadata filtering
# We can restrict the search.For example, if we want only chunks from page 4:

results = chromadb.similarity_search(
    query,
    k=3,
    filter={"page": 4}
)
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print()

predominantly Muslim and third where population of two communities would be nearly equal; The 
provincial representatives to the All-India Constituent Assembly, after a preliminary meeting of that body, 
would meet in three group assemblies to frame constitutions or their component provinces and, if desired, 
for their group as well. 
•Among these constitutions, would be distributed the functions of government other than the three
{'creator': 'Keynote', 'page': 4, 'title': 'THe Making of Indian Constitution I', 'page_label': '5', 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'moddate': "D:20260106092821Z00'00'", 'creationdate': "D:20260106092821Z00'00'", 'source': 'text.pdf', 'total_pages': 15}

Muslim League did not agree with the the idea, preferring British presence in India as a deterrent to 
Congress power, which they said would be synonymous with Hindu domination. 
•Congress demanded election of members to Assembly based on Adult Suffrage; Terming it cumbersom

In [31]:
# DELETE DOCUMENTS
# When we add a document, Chroma gives/stores an ID.
# We can explicitly create IDs:

ids=[f"chunk_{i}" for i in range(len(chunks))]
chromDB=Chroma.from_documents(documents=chunks,embedding=embeddings,ids=ids,collection_name="pdf_search")


In [32]:
chromDB.delete(ids=["chunk_0"])